In [1]:

import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import resnet18
from sklearn.metrics import r2_score
from torch.utils.data import random_split
import seaborn as sns
import matplotlib.pyplot as plt
import copy

BATCH_SIZE_SSL = 256
BATCH_SIZE_REG = 128
SSL_LR = 1e-4
EPOCHS_SSL = 250
EPOCHS_REG_HEAD = 100
EPOCHS_FINETUNE = 100
TEMPERATURE = 0.2
FEATURE_DIM = 256       # bigger projection dim
PATIENCE = 12
IMG_DIR = '/kaggle/input/csiro-biomass/'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
TARGETS_CALC=['GDM_g', 'Dry_Total_g']
LOCAL_RESNET="/kaggle/input/resnet18/pytorch/default/1/resnet18-5c106cde.pth"

# Load your training data
df = pd.read_csv("/kaggle/input/csiro-biomass/train.csv")
bio_scaler = StandardScaler()


# Convert long format to wide format
wide_df = df.pivot_table(
    index=["image_path", "Pre_GSHH_NDVI", "Height_Ave_cm", "Sampling_Date"],
    columns="target_name",
    values="target"
).reset_index()
wide_df.columns.name = None

# Drop unnecessary columns
wide_df = wide_df.drop(columns=['Sampling_Date'])  # omit date
tab_features = ['Pre_GSHH_NDVI', 'Height_Ave_cm']

# Add Month column from Sampling_Date before dropping
df['Sampling_Date'] = pd.to_datetime(df['Sampling_Date'], errors='coerce')
wide_df['Month'] = df['Sampling_Date'].dt.month
wide_df['Month'] = wide_df['Month'] / 12.0 


# Normalize tabular features (0-1)
for col in tab_features:
    wide_df[col] = (wide_df[col] - wide_df[col].min()) / (wide_df[col].max() - wide_df[col].min())

wide_df[TARGETS] = bio_scaler.fit_transform(wide_df[TARGETS])

print("Tabular features:", tab_features)
print("Targets (first 5 rows):\n", wide_df[TARGETS].head())



# -------------------------------
#  DATASET
# -------------------------------
class BiomassImageDataset(Dataset):
    def __init__(self, df, img_dir, tab_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tab_data = torch.tensor(df[tab_cols].values, dtype=torch.float32)
        self.transform = transform

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['image_path'])).convert('RGB')
        if self.transform:
            x1, x2 = self.transform(img)
        else:
            x1 = x2 = transforms.ToTensor()(img)
        tab = self.tab_data[idx]  # this is already a tensor
        return x1, x2, tab.float()  # ensure float tensor

    def __len__(self):
        return len(self.df)

v1 = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3, saturation=0.3, hue=0.02
    ),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

v2 = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0, translate=(0.05, 0.05), shear=5
    ),
    transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
# -------------------------------
# 2. Dual augmentation
# -------------------------------
class ContrastiveTransform:
    def __init__(self, img_size=224):
        self.t1 = v1
        self.t2 = v2

    def __call__(self, x):
        return self.t1(x), self.t2(x)

contrastive_transform = ContrastiveTransform()

ssl_dataset = BiomassImageDataset(wide_df, IMG_DIR,tab_features,transform=contrastive_transform)
ssl_loader = DataLoader(ssl_dataset, batch_size=BATCH_SIZE_SSL, shuffle=True, num_workers=4, pin_memory=True)

Tabular features: ['Pre_GSHH_NDVI', 'Height_Ave_cm']
Targets (first 5 rows):
    Dry_Green_g  Dry_Dead_g  Dry_Clover_g
0    -0.408018    1.611179     -0.549526
1    -0.750020   -0.972540     -0.549526
2    -1.049638   -0.972540     -0.049558
3    -0.094109    1.528165     -0.549526
4    -0.634663    0.902680     -0.513636


In [2]:
# -------------------------
# Model: ResNet18 backbone + projection MLP
# -------------------------
class SSLResNet(nn.Module):
    def __init__(self, local_weights_path=None, feature_dim=FEATURE_DIM):
        super().__init__()
        # load base resnet
        backbone = resnet18()
        # load state_dict from local file (offline)
        if local_weights_path and os.path.exists(local_weights_path):
            sd = torch.load(local_weights_path, map_location='cpu',weights_only=False)
            # handle possible "state_dict" key or raw dict
            if 'state_dict' in sd:
                sd = sd['state_dict']
            # if keys have "module." prefix, strip them
            new_sd = {}
            for k,v in sd.items():
                kk = k
                if k.startswith('module.'):
                    kk = k[len('module.'):]
                new_sd[kk] = v
            backbone.load_state_dict(new_sd, strict=False)
            print("Loaded local ResNet weights from", local_weights_path)
        else:
            print("Local resnet weights not found; using random init (offline mode).")
        # remove fc
        backbone.fc = nn.Identity()
        self.backbone = backbone
        # projection head: 512 -> 256 -> feature_dim
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, feature_dim)
        )

        self.aux_head = nn.Sequential(
            nn.Linear(512 + 2, 256),
            nn.ReLU(),
            nn.Linear(256, 2)   # predict tab features or targets
        )
    # In SSLResNet.forward
    def forward(self, img, tab=None, return_aux=False):
        h = self.backbone(img) # 512
        
        if return_aux and tab is not None:
            # This is fine; it checks if tab is not None before combining
            combined = torch.cat([h, tab], dim=1)
            aux = self.aux_head(combined)
            return h, aux
        
        return h, None # Returns features only
        
ssl_model = SSLResNet(local_weights_path=LOCAL_RESNET, feature_dim=FEATURE_DIM).to(DEVICE)


# -------------------------
# stable NT-Xent loss
# -------------------------
def nt_xent_loss(z_i, z_j, temperature=TEMPERATURE):
    z = torch.cat([z_i, z_j], dim=0)            # 2N x D
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.T) / temperature    # 2N x 2N
    N = z_i.size(0)
    labels = torch.arange(N, device=z.device)
    labels = torch.cat([labels, labels], dim=0)
    # mask to remove self-similarity
    mask = (~torch.eye(2*N, dtype=torch.bool, device=z.device)).float()
    exp_sim = torch.exp(sim) * mask
    # positive similarities: each i matches to i+N (and vice-versa)
    pos_indices = torch.cat([torch.arange(N, N*2, device=z.device), torch.arange(N, device=z.device)], dim=0)
    pos_sim = sim[torch.arange(2*N, device=z.device), pos_indices] 
    positives = torch.exp(pos_sim)
    denom = exp_sim.sum(dim=1)
    loss = -torch.log(positives / denom)
    return loss.mean()

# -------------------
# 4. Self-supervised training loop
# -------------------
optimizer_ssl = torch.optim.AdamW(ssl_model.parameters(), lr=SSL_LR, weight_decay=1e-6)
scheduler_ssl = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ssl, T_max=EPOCHS_SSL)

best_loss = float('inf')
counter = 0
for epoch in range(EPOCHS_SSL):
    ssl_model.train()
    total_loss = 0.0
    for x1, x2 ,tab in ssl_loader:
        x1, x2 , t= x1.to(DEVICE), x2.to(DEVICE), tab.to(DEVICE)
        (z1, aux1) = ssl_model(x1, t, return_aux=True)
        (z2, aux2) = ssl_model(x2, t, return_aux=True)
        ssl_loss  = nt_xent_loss(z1, z2, temperature=TEMPERATURE)
        aux_loss = F.mse_loss(aux1, t) + F.mse_loss(aux2, t)
        aux_loss = 1.0 * aux_loss
        loss = ssl_loss + aux_loss
        optimizer_ssl.zero_grad()
        loss.backward()
        optimizer_ssl.step()
        total_loss += loss.item()
    avg = total_loss / len(ssl_loader)
    print(f"SSL Epoch {epoch+1}/{EPOCHS_SSL}  avg_loss={avg:.4f}")
    if avg < best_loss:
        best_loss = avg
        counter = 0
        # save checkpoint
        torch.save(ssl_model.state_dict(), "ssl_backbone_best.pth")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("Early stopping SSL.")
            break
    scheduler_ssl.step()


# -------------------------
# Regression head (using backbone features)
# -------------------------
# Build a regression model that uses pretrained backbone (feature extractor)
class RegressionModel(nn.Module):
    def __init__(self, backbone, tab_dim ,output_dim=3):
        super().__init__()
        # backbone expected to output 512-d features
        self.backbone = backbone  # backbone should output 512 (we use backbone.backbone if using SSLResNet)
        # big head
        self.tab_dim = tab_dim 
        self.fc = nn.Sequential(
            nn.Linear(512+tab_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    # In RegressionModel.forward
    def forward(self, x, tab=None):
        # Get image features (512-dim)
        # If the backbone is the SSLResNet wrapper, we access its internal backbone.
        if isinstance(self.backbone, nn.Module) and hasattr(self.backbone, 'backbone') and isinstance(self.backbone.backbone, nn.Module):
            # This handles the SSLResNet structure correctly, calling the inner ResNet
            h = self.backbone.backbone(x) 
        else:
            # Fallback for a regular backbone
            h = self.backbone(x)
    
        # --- concatenate tab after the backbone ---
        # Ensure tab is available for the head, even if missing (using zeros)
        if tab is None:
            tab = torch.zeros(x.size(0), self.tab_dim, device=x.device)
    
        combined = torch.cat([h, tab], dim=1)
        out = self.fc(combined)
        return out
        
# instantiate regression model and freeze backbone initially
reg_model = RegressionModel(ssl_model,2).to(DEVICE)
for param in reg_model.backbone.parameters():
    param.requires_grad = False

# regression dataset (images + targets)
class BiomassRegressionDataset(Dataset):
    def __init__(self, df, img_dir, targets, tab_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.targets = torch.tensor(df[targets].values, dtype=torch.float32)
        self.tabular = torch.tensor(df[tab_cols].values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_path'])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.tabular[idx], self.targets[idx]

reg_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
reg_dataset = BiomassRegressionDataset(
    wide_df, IMG_DIR, TARGETS, tab_features, transform=reg_transform
)
reg_loader = DataLoader(reg_dataset, batch_size=BATCH_SIZE_REG, shuffle=True, num_workers=4, pin_memory=True)

# optimizer for head only
optimizer_reg = torch.optim.AdamW(filter(lambda p: p.requires_grad, reg_model.parameters()), lr=1e-3, weight_decay=1e-6)
criterion = nn.MSELoss()

best_reg_loss = float('inf')
counter = 0
for epoch in range(EPOCHS_REG_HEAD):
    reg_model.train()
    tot = 0.0
    for imgs, tab, targets in reg_loader:
        imgs, tab, targets = imgs.to(DEVICE), tab.to(DEVICE), targets.to(DEVICE)
        preds = reg_model(imgs,tab)
        loss = criterion(preds, targets)
        optimizer_reg.zero_grad()
        loss.backward()
        optimizer_reg.step()
        tot += loss.item()
    avg = tot / len(reg_loader)
    print(f"Reg head epoch {epoch+1}/{EPOCHS_REG_HEAD}  loss={avg:.4f}")
    if avg < best_reg_loss:
        best_reg_loss = avg
        counter = 0
        torch.save(reg_model.state_dict(), "reg_head_best.pth")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("Early stopping reg head.")
            break

# -------------------------
# Fine-tune backbone: unfreeze layer3 & layer4
# -------------------------
print("Unfreezing backbone layers layer3 & layer4 for fine-tuning")
for name, param in reg_model.backbone.backbone.named_parameters():
    if name.startswith('layer3') or name.startswith('layer4') or name.startswith('layer2'):  # you can choose layer2 too
        param.requires_grad = True
    else:
        param.requires_grad = False

# set differential learning rates: small for backbone, larger for head
backbone_params = [p for n,p in reg_model.backbone.backbone.named_parameters() if p.requires_grad]
head_params = [p for p in reg_model.fc.parameters()]

optimizer_ft = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 3e-4},
    {'params': head_params, 'lr': 1e-3}
], weight_decay=1e-4)

scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=EPOCHS_FINETUNE)

best_ft = float('inf')
counter = 0
for epoch in range(EPOCHS_FINETUNE):
    reg_model.train()
    tot = 0.0
    for imgs, tab, targets in reg_loader:
        imgs, tab, targets = imgs.to(DEVICE), tab.to(DEVICE), targets.to(DEVICE)
        preds = reg_model(imgs,tab)
        loss = criterion(preds, targets)
        optimizer_ft.zero_grad()
        loss.backward()
        optimizer_ft.step()
        tot += loss.item()
    avg = tot / len(reg_loader)
    print(f"Finetune epoch {epoch+1}/{EPOCHS_FINETUNE}  loss={avg:.4f}")
    if avg < best_ft:
        best_ft = avg
        counter = 0
        torch.save(reg_model.state_dict(), "reg_finetuned_best.pth")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("Early stopping finetune.")
            break
    scheduler_ft.step()

print("Training finished. Best finetune loss:", best_ft)
# Save final models
torch.save(ssl_model.state_dict(), "ssl_backbone_final.pth")
torch.save(reg_model.state_dict(), "reg_model_final.pth")

Loaded local ResNet weights from /kaggle/input/resnet18/pytorch/default/1/resnet18-5c106cde.pth
SSL Epoch 1/250  avg_loss=5.5599
SSL Epoch 2/250  avg_loss=5.1222
SSL Epoch 3/250  avg_loss=4.8728
SSL Epoch 4/250  avg_loss=4.7191
SSL Epoch 5/250  avg_loss=4.4712
SSL Epoch 6/250  avg_loss=4.2565
SSL Epoch 7/250  avg_loss=4.1236
SSL Epoch 8/250  avg_loss=4.0092
SSL Epoch 9/250  avg_loss=3.8952
SSL Epoch 10/250  avg_loss=3.7875
SSL Epoch 11/250  avg_loss=3.7060
SSL Epoch 12/250  avg_loss=3.6562
SSL Epoch 13/250  avg_loss=3.5828
SSL Epoch 14/250  avg_loss=3.5179
SSL Epoch 15/250  avg_loss=3.4558
SSL Epoch 16/250  avg_loss=3.4087
SSL Epoch 17/250  avg_loss=3.3648
SSL Epoch 18/250  avg_loss=3.3558
SSL Epoch 19/250  avg_loss=3.2725
SSL Epoch 20/250  avg_loss=3.2592
SSL Epoch 21/250  avg_loss=3.2250
SSL Epoch 22/250  avg_loss=3.2359
SSL Epoch 23/250  avg_loss=3.1724
SSL Epoch 24/250  avg_loss=3.1757
SSL Epoch 25/250  avg_loss=3.1232
SSL Epoch 26/250  avg_loss=3.1050
SSL Epoch 27/250  avg_loss=3.

In [3]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_path'])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, row['image_path']


# Load test CSV
test_df = pd.read_csv("/kaggle/input/csiro-biomass/test.csv")
test_df = test_df.drop_duplicates(subset=["image_path"]).reset_index(drop=True)

test_dataset = TestDataset(test_df, IMG_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE_REG, shuffle=False)


# -----------------------------
# 2. Inference
# -----------------------------
reg_model.eval()
results = []

with torch.no_grad():
    for images, paths in test_loader:
        images = images.to(DEVICE)

        preds = reg_model(images)        # scaled predictions
        preds = preds.cpu().numpy()

        # invert normalization to real biomass units
        pred_real = bio_scaler.inverse_transform(preds)
        pred_real = np.maximum(0, pred_real)
        dry_green  = pred_real[:, 0]
        dry_dead   = pred_real[:, 1]
        dry_clover = pred_real[:, 2]

        gdm       = dry_clover + dry_green
        dry_total = dry_clover + dry_dead + dry_green

        # build submission rows
        for i, path in enumerate(paths):
            base_id = os.path.splitext(os.path.basename(path))[0]

            results.extend([
                {'sample_id': f"{base_id}__Dry_Clover_g", 'target': float(dry_clover[i])},
                {'sample_id': f"{base_id}__Dry_Dead_g",   'target': float(dry_dead[i])},
                {'sample_id': f"{base_id}__Dry_Green_g",  'target': float(dry_green[i])},
                {'sample_id': f"{base_id}__Dry_Total_g",  'target': float(dry_total[i])},
                {'sample_id': f"{base_id}__GDM_g",        'target': float(gdm[i])},
            ])

# -----------------------------
# 3. Save CSV for Kaggle submission
# -----------------------------
submission_df = pd.DataFrame(results)
submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission saved to /kaggle/working/submission.csv")
print(submission_df.head())

Submission saved to /kaggle/working/submission.csv
                    sample_id     target
0  ID1001187975__Dry_Clover_g   2.538090
1    ID1001187975__Dry_Dead_g  16.575993
2   ID1001187975__Dry_Green_g  20.813799
3   ID1001187975__Dry_Total_g  39.927879
4         ID1001187975__GDM_g  23.351889
